# Сквозной пайплайн: рецидивирующее течение ДКА

Ноутбук прогоняется сверху вниз на исправленной врачами базе. Логика вынесена в пакет `src`, здесь только вызовы, вывод и выводы по шагам. Таблицы пишем в `reports/tables`, рисунки в `reports/figures`.

In [ ]:
import itertools
import os
import sys
from pathlib import Path

# без полос прогресса SDV и TabPFN в выводе: tqdm читает переменную при импорте
os.environ["TQDM_DISABLE"] = "1"

# чтобы импортировать пакет src независимо от того, откуда запущен jupyter
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd

from src import config, data, eda, evaluate, finalist, importance, interactions, models, screening, selection, stats, structure, synthetic, thresholds, tuning

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
config.ensure_dirs()


def save_table(table, name):
    table.to_csv(config.TABLES_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")


def save_figure(fig, name):
    fig.savefig(config.FIGURES_DIR / f"{name}.png", dpi=config.FIGURE_DPI, bbox_inches="tight")


figure_numbers = itertools.count(1)


def caption(fig, text):
    # номер рисунка идет по порядку ячеек, вставка нового рисунка не сбивает нумерацию
    number = next(figure_numbers)
    eda.add_caption(fig, f"Рисунок {number}. {text}")
    print(f"рисунок {number}")

## Шаг 0. Данные и разведочный анализ

### 0.1 Загрузка и приведение типов

Врачи исправили базу сами, поэтому очистка только приводит типы: числа из текста, длительность СД в годах, стадии ХБП к единому виду. Строки не удаляем.

In [ ]:
raw = data.load_raw()
df = data.clean(raw)
labeled = data.labeled(df)
print(f"строк в базе {len(raw)}, колонок {raw.shape[1]}, с исходом {len(labeled)}")
labeled[config.TARGET].value_counts().rename(index=eda.GROUP_NAMES)

### 0.2 Словарь колонок

Роль каждой колонки: количественный, бинарный, порядковый, номинальный признак или исключен с причиной. Тип решает кодирование: количественные и порядковые идут в модель одним числом, OHE получают только номинальные.

In [ ]:
dictionary = eda.column_dictionary(raw, df)
save_table(dictionary, "step0_columns")
dictionary

In [ ]:
dictionary["роль"].value_counts()

### 0.3 Пропуски в признаках модели

In [ ]:
fig = eda.plot_missing(df, config.FEATURES)
caption(fig, f"Доля пропусков по {len(config.FEATURES)} признакам модели у {len(labeled)} пациентов с известным исходом.")
save_figure(fig, "step0_missing")
plt.show()

### 0.4 Количественные признаки по исходу

In [ ]:
fig = eda.plot_distributions(df, config.QUANTITATIVE)
caption(fig, "Распределения количественных признаков по исходу: синий - единичный эпизод, оранжевый - рецидив. По оси y плотность, чтобы группы разного размера сравнивались по форме, n - число пациентов с известным значением.")
save_figure(fig, "step0_distributions")
plt.show()

In [ ]:
fig = eda.plot_boxplots(df, config.QUANTITATIVE)
caption(fig, "Количественные признаки по исходу: линия - медиана, ящик - квартили, усы - 1,5 межквартильного размаха, точки - значения за усами. Под ящиком число пациентов с известным значением.")
save_figure(fig, "step0_boxplots")
plt.show()

### 0.5 Категориальные признаки по исходу

In [ ]:
fig = eda.plot_category_rates(df, config.CATEGORICAL)
caption(fig, "Доля рецидива по уровням категориальных признаков, пропуск показан отдельным уровнем. Над столбцом число пациентов уровня, пунктир - доля рецидива во всей выборке.")
save_figure(fig, "step0_categories")
plt.show()

### 0.6 Корреляции и мультиколлинеарность

Возраст манифестации и ЛПНП оставлены на тепловой карте, чтобы показать, почему они исключены. Попарная корреляция не видит тройную зависимость (возраст = манифестация + длительность, ОХ = ЛПНП + ЛПВП + ТГ / 2,2), ее видит VIF.

In [ ]:
fig = eda.plot_correlation(labeled, config.QUANTITATIVE + [config.MANIFESTATION, config.LDL])
caption(fig, "Корреляции Спирмена количественных признаков, включая исключенные из модели возраст манифестации и ЛПНП. Каждая пара считается по пациентам, у которых известны оба значения.")
save_figure(fig, "step0_correlation")
plt.show()

In [ ]:
# VIF по клиническим блокам: на всех 16 признаках разом полных строк почти нет
blocks = {
    "возраст с манифестацией": ["Возраст (на текущий момент)", config.DURATION, config.MANIFESTATION],
    "возраст без манифестации": ["Возраст (на текущий момент)", config.DURATION],
    "липиды с ЛПНП": [config.TOTAL_CHOLESTEROL, config.LDL, config.HDL, config.TRIGLYCERIDES],
    "липиды без ЛПНП": [config.TOTAL_CHOLESTEROL, config.HDL, config.TRIGLYCERIDES],
    "кислотно-основное состояние": [
        "pH при поступлении", "ВЕ при поступлении", "Лактат при поступлении", "Глюкоза при поступлении",
    ],
    "почки": ["Креатинин при поступлении", "Мочевина при поступлении"],
}
vif = pd.concat(
    [eda.vif(labeled, cols).assign(блок=name) for name, cols in blocks.items()],
    ignore_index=True,
)
save_table(vif, "step0_vif")
vif

### 0.7 ЛПНП рассчитан по формуле Фридвальда

Если ЛПНП получен расчетом, общий холестерин точно равен ЛПНП + ЛПВП + ТГ / 2,2 (ммоль/л), и один из четырех липидов не несет новой информации. Проверяем двумя способами: подставляем значения в формулу и подбираем коэффициенты регрессией. При расчетном ЛПНП коэффициенты регрессии должны совпасть с формулой, а R² быть близок к 1.

In [ ]:
friedewald, mismatches = eda.friedewald_check(labeled)
print(f"пациентов с четырьмя липидами {friedewald.attrs['n']}, "
      f"формула выполняется с точностью до {eda.FRIEDEWALD_TOLERANCE} ммоль/л у {friedewald.attrs['совпало']}")
save_table(friedewald, "step0_friedewald")
friedewald

In [ ]:
# пациенты, у которых формула не выполняется: ЛПНП у них, вероятно, измерен напрямую
mismatches

In [ ]:
fig = eda.plot_excluded(eda.excluded_pairs(labeled))
caption(fig, "Исключенные признаки против их выражения через признаки модели: возраст манифестации против возраста минус длительность СД, ЛПНП против ОХ - ЛПВП - ТГ / 2,2 (формула Фридвальда). Пунктир - полное совпадение, в заголовке R² и число пациентов с известными значениями.")
save_figure(fig, "step0_excluded")
plt.show()

### Вывод шага 0

- В базе 303 строки, исход известен у 287: 198 единичных эпизодов и 89 рецидивов (31%).
- В модель идут 24 признака: 15 количественных, 3 бинарных, 4 порядковых, 2 номинальных. 12 колонок исключены с причиной, 11 пустых удалены.
- Больше всего пропусков у HbA1c (60%), натрия (59%), калия (55%), лактата (49%), липидов (44-46%) и ХБП А (43%). У количественных доля рецидива среди пропусков и заполненных близка, 26-35%.
- Исключение - тип СД: не указан у 28 пациентов, из них 19 с рецидивом (68% против 27% среди заполненных). Пропуск здесь связан с исходом. Вопрос врачам: тип СД, скорее всего, можно восстановить по историям.
- Вид инсулинотерапии 0 у 59 пациентов, рецидив у одного, у 47 из них длительность СД 0.
- Возраст, возраст манифестации и длительность СД дают VIF 76, 85 и 34, без манифестации 1,1. Манифестацию исключаем.
- ЛПНП рассчитан по формуле Фридвальда: у 144 из 154 пациентов с четырьмя липидами общий холестерин равен ЛПНП + ЛПВП + ТГ / 2,2 с точностью до 0,1 ммоль/л. Регрессия восстанавливает коэффициенты формулы: 0,97, 0,89 и 0,44 против 1, 1 и 0,455, R² 0,949. Ниже 1 его опускают 10 пациентов с прямым измерением или ошибкой, у двух расхождение 2,4 и 2,6 ммоль/л. Зависимость близка к полной: VIF общего холестерина 19,6, ЛПНП 13,1, без ЛПНП все три липида ниже 1,6. ЛПНП исключаем как расчетный.
- Самые сильные корреляции Спирмена: pH и BE 0,88, общий холестерин и ЛПНП 0,83, креатинин и мочевина 0,67, креатинин и глюкоза 0,65.

## Шаг 1. Статистика по протоколу

Описание когорты, в отбор признаков не идет. Критерии выбираются по протоколу врачей (`src/stats.py`), значимость по сырому p < 0,05 без поправки на множественность, значимый p помечен звездочкой.

### 1.1 Рецидив против единичного эпизода

In [ ]:
single = labeled[config.TARGET] == 0
recurrent = labeled[config.TARGET] == 1

t1 = stats.two_groups_quantitative(labeled, config.STATS_QUANTITATIVE, single, recurrent)
save_table(t1, "step1_recurrence_quantitative")
t1

In [ ]:
t2 = stats.two_groups_categorical(labeled, config.STATS_CATEGORICAL, single, recurrent)
save_table(t2, "step1_recurrence_categorical")
t2

In [ ]:
t3 = stats.describe_quantitative(labeled, config.STATS_QUANTITATIVE)
t4 = stats.describe_categorical(labeled, config.STATS_CATEGORICAL)
save_table(t3, "step1_describe_quantitative")
save_table(t4, "step1_describe_categorical")
t3

In [ ]:
t4

### 1.2 Единичные без дебюта против всех рецидивов

Запрос врачей: длительность СД, суточная доза и вид инсулинотерапии, из единичных исключены пациенты с дебютом. У 10 пациентов длительность СД неизвестна, их нельзя отнести ни к дебюту, ни к стажу. Поэтому "без дебюта" считаем в двух вариантах:

- строгий (`known`): длительность известна и больше 0, неизвестная исключается;
- как в документах для врачей от 22 сентября (`with_unknown`): исключается только длительность 0, неизвестная остается.

In [ ]:
duration = labeled[config.DURATION]
no_debut_variants = {
    "known": duration > 0,  # пропуск дает False и исключается
    "with_unknown": duration != 0,  # пропуск дает True и остается
}
print(f"длительность 0: {(duration == 0).sum()} (рецидивов среди них {(recurrent & (duration == 0)).sum()}), "
      f"неизвестна: {duration.isna().sum()} (рецидивов {(recurrent & duration.isna()).sum()})")

subgroup_columns = [config.DURATION, "Суточная доза инсулина"]
therapy = ["Вид инсулинотерапии (1 - ручки, 2 - помпа) на момент ДКА"]
for variant, mask in no_debut_variants.items():
    single_no_debut = single & mask
    subgroup = pd.concat(
        [
            stats.two_groups_quantitative(labeled, subgroup_columns, single_no_debut, recurrent),
            stats.two_groups_categorical(labeled, therapy, single_no_debut, recurrent),
        ]
    )
    save_table(subgroup, f"step1_subgroup_no_debut_{variant}")
    print(f"вариант {variant}: единичных {single_no_debut.sum()} против рецидивов {recurrent.sum()}")
    display(subgroup)

### 1.3 Рецидив против единичного эпизода без дебюта, оба варианта

In [ ]:
no_debut_tables = {}
for variant, mask in no_debut_variants.items():
    cohort = labeled[mask]
    single_nd, recurrent_nd = cohort[config.TARGET] == 0, cohort[config.TARGET] == 1
    quantitative = stats.two_groups_quantitative(cohort, config.STATS_QUANTITATIVE, single_nd, recurrent_nd)
    categorical = stats.two_groups_categorical(cohort, config.STATS_CATEGORICAL, single_nd, recurrent_nd)
    save_table(quantitative, f"step1_no_debut_{variant}_quantitative")
    save_table(categorical, f"step1_no_debut_{variant}_categorical")
    no_debut_tables[variant] = (quantitative, categorical)
    print(f"вариант {variant}: пациентов {len(cohort)}, единичных {single_nd.sum()}, рецидивов {recurrent_nd.sum()}")

In [ ]:
no_debut_tables["known"][0]

In [ ]:
no_debut_tables["known"][1]

In [ ]:
no_debut_tables["with_unknown"][0]

In [ ]:
no_debut_tables["with_unknown"][1]

### 1.4 Степень тяжести ДКА (три группы)

Все пациенты с указанной степенью тяжести, исход для этой таблицы не нужен.

In [ ]:
levels = [1.0, 2.0, 3.0]
severity = df[df[config.SEVERITY].isin(levels)]
print("пациентов по степени:", severity[config.SEVERITY].value_counts().sort_index().to_dict())

categorical_sev = [c for c in config.STATS_CATEGORICAL if c != config.SEVERITY]
s1 = stats.three_groups_quantitative(severity, config.STATS_QUANTITATIVE, config.SEVERITY, levels)
s2 = stats.three_groups_categorical(severity, categorical_sev, config.SEVERITY, levels)
save_table(s1, "step1_severity_quantitative")
save_table(s2, "step1_severity_categorical")
s1

In [ ]:
s2

In [ ]:
severity_duration = severity[config.DURATION]
severity_tables = {}
for variant, mask in {"known": severity_duration > 0, "with_unknown": severity_duration != 0}.items():
    cohort = severity[mask]
    quantitative = stats.three_groups_quantitative(cohort, config.STATS_QUANTITATIVE, config.SEVERITY, levels)
    categorical = stats.three_groups_categorical(cohort, categorical_sev, config.SEVERITY, levels)
    save_table(quantitative, f"step1_severity_no_debut_{variant}_quantitative")
    save_table(categorical, f"step1_severity_no_debut_{variant}_categorical")
    severity_tables[variant] = (quantitative, categorical)
    print(f"вариант {variant}: по степени {cohort[config.SEVERITY].value_counts().sort_index().to_dict()}")

In [ ]:
severity_tables["known"][0]

In [ ]:
severity_tables["known"][1]

In [ ]:
severity_tables["with_unknown"][0]

In [ ]:
severity_tables["with_unknown"][1]

### Вывод шага 1

Таблицы по всем пациентам и варианты без дебюта `with_unknown` совпадают с документами для врачей от 22 сентября.

- Рецидив против единичного эпизода, 287 пациентов. Различаются:
  - длительность СД, Me 9 против 6 лет, p = 0,001;
  - суточная доза инсулина, 43 против 32 ЕД, p < 0,001;
  - HbA1c, 10,99 против 12,22%, p = 0,006, в группе рецидива ниже;
  - ЛПВП, 1,26 против 1,00, p = 0,001;
  - вид инсулинотерапии, ретинопатия, ХБП С и ХБП А: p от < 0,001 до 0,008;
  - нейропатия: ОШ 2,19 (1,25; 3,83), p = 0,006;
  - алкоголь: ОШ 2,10 (1,15; 3,83), p = 0,014.

  Количество ДКА в анамнезе различается по построению исхода и в модель не идет.
- Без дебюта. Все 54 пациента с длительностью 0 из группы единичных, у 10 длительность неизвестна (3 рецидива). Оба варианта дают одни выводы:

  | | `known` (длительность > 0) | `with_unknown` (длительность ≠ 0) |
  |---|---|---|
  | пациентов (единичных / рецидивов) | 223 (137 / 86) | 233 (144 / 89) |
  | ЛПВП, p | 0,018 | 0,031 |
  | вид инсулинотерапии, p | 0,018 | 0,021 |
  | алкоголь, ОШ и p | 1,47 (0,78; 2,78), p = 0,235 | 1,47 (0,79; 2,71), p = 0,219 |
  | единичные без дебюта против всех рецидивов: доза, p | 0,093 | 0,061 |
  | то же: вид инсулинотерапии, p | 0,014 | 0,021 |

  Остальные различия исчезают, длительность в подгруппе p = 0,465 в обоих вариантах.
- Степень тяжести, 257 пациентов (38, 78 и 141). Различаются показатели тяжести самого эпизода (pH, BE, калий, глюкоза, креатинин, мочевина), суточная доза (p = 0,014), ЛПВП, ТГ, вид инсулинотерапии, нейропатия (p = 0,014). Без дебюта доза и нейропатия перестают различаться, появляется возраст манифестации (p = 0,012 в обоих вариантах). ТГ различаются в варианте `with_unknown` (206 пациентов, p = 0,035) и не различаются в `known` (196 пациентов).
- Поправку на множественность протокол не предусматривает. При таком числе сравнений вторичные различия трактуем как разведочные.

## Шаг 2. Схема валидации

На этом шаге ничего не обучается: разбиваем данные и фиксируем, как оцениваем все модели дальше.

Разбиваем один раз: стратифицированно, 20% в тест, seed 0. Состав теста закреплен списком N пп в `config.TEST_IDS`, чтобы правки базы врачами его не перемешали. Тест не трогаем до выбора финальных моделей (шаг 11).

Все модели проходят одну процедуру (`src/evaluate.py`):
- внешняя стратифицированная CV, 5 повторов по 5 фолдов, одни и те же фолды для всех моделей. Повторы нужны, потому что одно разбиение шумит (разброс ROC-AUC по фолдам - в шаге 3);
- если модель подбирает гиперпараметры или отбирает признаки, это делается внутри каждого внешнего фолда (вложенная CV). Решения, принятые сравнением вариантов на этих же фолдах (импутация, балансировка), в оценку не вложены, и она немного завышена;
- в каждом повторе каждый пациент предсказан моделью, которая его не видела (OOF). ROC-AUC, PR-AUC, калибровку и матрицу ошибок считаем по каждому повтору и усредняем, ROC-AUC еще и по 25 фолдам;
- Brier сравниваем с эталоном: постоянный прогноз, равный доле рецидива. Масштабированный Brier: 0 - модель не лучше постоянного прогноза, меньше 0 - хуже;
- калибровка: наклон (1 - разброс прогнозов верный, меньше 1 - прогнозы слишком крайние, больше 1 - слишком сжаты к средней доле) и калибровка в среднем (0 - средний прогноз равен доле рецидива);
- рабочая точка: в каждом повторе подбираем порог (основной - максимум F1, рядом максимум F2, для сравнения Юден) и считаем матрицу ошибок, долю помеченных, чувствительность, специфичность, PPV и NPV. Порог подобран по тем же прогнозам, поэтому числа оптимистичны;
- кривая принятия решений (DCA): чистая польза модели при разных порогах против стратегий «пометить всех» и «никого»;
- графики строим по всем повторам, сложенным вместе;
- бутстрэп-ДИ отражают разброс пациентов при фиксированных прогнозах, изменчивость обучения в них не входит.

**Сравнение вариантов.** Смотрим ROC-AUC OOF с 95% бутстрэп-ДИ. Если ДИ перекрываются, лучший вариант по этим данным выявить нельзя, и берем более простой.

In [ ]:
X, y = data.model_frame(df)
X_train, X_test, y_train, y_test = evaluate.split(X, y)  # X_test и y_test не трогаем до шага 11
print(f"признаков {X.shape[1]}, train {len(y_train)} (рецидивов {y_train.sum()}), "
      f"test {len(y_test)} (рецидивов {y_test.sum()})")

## Шаг 3. Бейзлайн

Самый простой стандартный вариант: логистическая регрессия с параметрами по умолчанию (L2, C = 1) на всех 24 признаках. Пропуски числовых признаков заполняются медианой, номинальных - самой частой категорией, числовые масштабируются, номинальные кодируются OHE. Это планка, которую должны обойти следующие шаги.

In [ ]:
baseline = evaluate.cross_validate("логрег, бейзлайн", models.baseline_logreg(), X_train, y_train)
fold_aucs = baseline["ROC-AUC по фолдам"]
print(f"ROC-AUC по {len(fold_aucs)} фолдам: от {min(fold_aucs):.3f} до {max(fold_aucs):.3f}")
results = [baseline]
summary = evaluate.summary_table(results).round(3)
save_table(summary, "step3_baseline")
summary.T

In [ ]:
# рабочие точки: порог подбирается в каждом повторе по OOF, метрики усреднены по повторам
points = evaluate.operating_points(baseline).round(3)
save_table(points, "step3_operating_points")
points.T

In [ ]:
fig = evaluate.plot_curves(results)
caption(fig, f"ROC- и PR-кривые бейзлайна по OOF-прогнозам на train ({len(y_train)} пациентов, {y_train.sum()} рецидивов), все 5 повторов CV вместе. Пунктир слева - случайное угадывание, справа - доля рецидива.")
save_figure(fig, "step3_curves")
plt.show()

In [ ]:
fig = evaluate.plot_calibration(results)
caption(fig, "Калибровка бейзлайна: средний прогноз и наблюдаемая доля рецидива в 5 квантильных группах OOF-прогнозов всех 5 повторов. Пунктир - идеальная калибровка.")
save_figure(fig, "step3_calibration")
plt.show()

In [ ]:
fig = evaluate.plot_dca(results)
caption(fig, "Кривая принятия решений: чистая польза модели при разных порогах вероятности против стратегий «пометить всех» и «никого». Модель полезна там, где ее кривая выше обеих.")
save_figure(fig, "step3_dca")
plt.show()
pd.DataFrame([evaluate.useful_thresholds(r) for r in results]).round(2)

### Вывод шагов 2 и 3

- Схема валидации одна для всех моделей: 229 пациентов train (71 рецидив), 5 повторов по 5 фолдов. Тест (58 пациентов) закрыт.
- Бейзлайн - логрег на всех 24 признаках, L2 с C = 1, медиана и мода, без подбора. ROC-AUC CV 0,682 ± 0,083, по 25 фолдам от 0,502 до 0,841: одно разбиение на 5 фолдов дало бы любое число из этого разброса, поэтому повторы нужны.
- ROC-AUC OOF 0,681 (95% ДИ 0,614-0,750), по повторам 0,666-0,689. PR-AUC 0,474 (0,380-0,589) при доле рецидива 0,31 (рисунок 7).
- Калибровка (рисунок 8): наклон 0,442, прогнозы слишком крайние. На 24 признаках при 71 рецидиве модель переобучается. Brier 0,211 против 0,214 у постоянного прогноза, масштабированный Brier 0,012: вероятности почти не точнее доли рецидива.
- Рабочие точки (порог, помечено, чувствительность, специфичность, PPV, NPV):
  - максимум F1: 0,127; 68%; 0,868; 0,410; 0,400; 0,877;
  - максимум F2: 0,062; 80%; 0,958; 0,276; 0,374; 0,940.
- Кривая принятия решений выше обеих стратегий в основном при порогах 0,16-0,45 (рисунок 9).
- Бейзлайн - точка отсчета. Каждый следующий шаг сравниваем с ним по ДИ ROC-AUC OOF.

## Шаг 4. Пропуски и структура данных

### 4.1 Сравнение способов импутации

Кандидаты: KNN (5 соседей), MICE (последовательная регрессия признаков друг на друга) и медиана. Номинальные во всех вариантах заполняются модой. Каждый вариант идет в бейзлайн-логрег и проходит ту же рамку оценки.

In [ ]:
imputation_results = [
    evaluate.cross_validate(f"логрег, {method}", models.baseline_logreg(method), X_train, y_train)
    for method in models.IMPUTATION_METHODS
]
imputation_summary = evaluate.summary_table(imputation_results)
save_table(imputation_summary.round(3), "step4_imputation")
imputation_summary.round(3).T

### 4.2 Откуда выигрыш медианы

Две проверки, в выбор метода они не входят:
- KNN только для количественных, бинарные и порядковые медианой: если выигрыш дают бинарные и порядковые, качество приблизится к медиане;
- KNN с индикаторами пропусков (столбец 0/1 "значение было пропущено" для каждого признака с пропусками): если выигрыш дает сам факт пропуска, индикаторы его покажут.

И описание: связан ли пропуск в каждом признаке с рецидивом (по протоколу, хи-квадрат Пирсона или точный критерий Фишера).

In [ ]:
check_results = [
    evaluate.cross_validate(f"логрег, {method}", models.baseline_logreg(method), X_train, y_train)
    for method in models.IMPUTATION_CHECKS
]
check_summary = evaluate.summary_table(imputation_results + check_results)
save_table(check_summary.round(3), "step4_imputation_checks")
check_summary.round(3).T

In [ ]:
fig = evaluate.plot_cv_auc(check_summary)
caption(fig, "ROC-AUC OOF бейзлайн-логрега с 95% бутстрэп-ДИ при трех способах импутации и двух проверочных вариантах KNN.")
save_figure(fig, "step4_imputation")
plt.show()

In [ ]:
missing_table = structure.missing_outcome(X_train, y_train)
save_table(missing_table.round(3), "step4_missing_outcome")
fig = structure.plot_missing_outcome(missing_table, y_train.mean())
caption(fig, f"Доля рецидива у пациентов train с пропуском в признаке и без него. В подписи признака p по протоколу (хи-квадрат Пирсона или точный критерий Фишера). Пунктир - доля рецидива на train ({stats.num(y_train.mean(), 2)}).")
save_figure(fig, "step4_missing_outcome")
plt.show()
missing_table.round(3)

### 4.3 Сохранение распределений

Статистика Колмогорова-Смирнова между заполненными и наблюдаемыми значениями каждого количественного признака на train, чем меньше, тем лучше.

In [ ]:
ks = structure.ks_table(X_train, models.IMPUTATION_METHODS)
save_table(ks.round(3).reset_index(names="признак"), "step4_ks")
ks.round(3)

In [ ]:
fig = structure.plot_pca_clouds(X_train, models.IMPUTATION_METHODS)
caption(fig, f"Пациенты train ({len(y_train)}) на плоскости двух главных компонент после трех способов импутации. Цвет - число заполненных значений у пациента: пациенты с заполненными значениями не должны уходить в отдельное облако.")
save_figure(fig, "step4_pca_clouds")
plt.show()

In [ ]:
chosen = "медиана"  # ДИ ROC-AUC трех методов перекрываются, берем самый простой (вывод шага 4)

### 4.4 Структура данных после выбранной импутации

PCA, t-SNE и k-means не работают с пропусками, а пациентов train без единого пропуска почти нет, поэтому считаем на заполненных данных. Числовые признаки train после масштаба и выбранной импутации (медиана): модель видит данные именно так. В облаках PCA заполненные пациенты не уходят в отдельное облако ни при одном методе. Номинальные в анализ без учителя не входят.

In [ ]:
Z = structure.imputed(X_train, chosen)
variance = structure.pca_variance(Z)
save_table(variance.round(3), "step4_pca_variance")
fig = structure.plot_pca_variance(variance)
caption(fig, "Доля дисперсии числовых признаков train по главным компонентам (столбцы) и накопленная доля (линия), пунктир - 80%. Импутация медианой.")
save_figure(fig, "step4_pca_variance")
plt.show()
print("компонент для 80% дисперсии:", int((variance["накопленная"] < 0.8).sum()) + 1, "из", len(variance))

In [ ]:
from sklearn.decomposition import PCA

pca_points = PCA(n_components=2, random_state=config.RANDOM_SEED).fit_transform(Z)
fig = structure.plot_by_outcome(pca_points, y_train, "PCA, первые две компоненты", "ГК")
caption(fig, "Пациенты train на плоскости двух главных компонент, цвет - исход. Импутация медианой.")
save_figure(fig, "step4_pca_outcome")
plt.show()

In [ ]:
fig = structure.plot_by_outcome(structure.tsne_points(Z), y_train, "t-SNE", "t-SNE")
caption(fig, f"Проекция t-SNE пациентов train (perplexity {structure.TSNE_PERPLEXITY}), цвет - исход. На t-SNE смысл имеет только соседство точек, расстояния между группами точек смысла не несут.")
save_figure(fig, "step4_tsne")
plt.show()

In [ ]:
silhouettes = structure.silhouette_table(Z)
save_table(silhouettes.round(3), "step4_silhouette")
fig = structure.plot_silhouette(silhouettes)
caption(fig, "Средний силуэт k-means при числе кластеров от 2 до 6: от -1 до 1, чем выше, тем четче кластеры отделены. Значения около 0,1 означают слабое разделение.")
save_figure(fig, "step4_silhouette")
plt.show()

In [ ]:
best_k = int(silhouettes.loc[silhouettes["силуэт"].idxmax(), "кластеров"])
cluster_labels = structure.kmeans_labels(Z, best_k)
clusters = structure.cluster_outcome(cluster_labels, y_train)
save_table(clusters.round(3).reset_index(), "step4_clusters")
print(f"лучшее число кластеров по силуэту: {best_k}, связь кластера с исходом: "
      f"{clusters.attrs['критерий']}, p = {clusters.attrs['p']:.3f}")
fig = structure.plot_clusters(pca_points, cluster_labels, clusters)
caption(fig, f"Кластеры k-means (k = {best_k}) на плоскости двух главных компонент, в легенде размер кластера и доля рецидива. Связь кластера с исходом: {clusters.attrs['критерий']}, p = {stats.num(clusters.attrs['p'], 3)}.")
save_figure(fig, "step4_clusters")
plt.show()
clusters.round(3)

### Вывод шага 4

- Импутация. ROC-AUC OOF: медиана 0,681 (95% ДИ 0,614-0,750), KNN 0,633 (0,564-0,702), MICE 0,632 (0,562-0,699), рисунок 10. ДИ перекрываются, лучший метод по этим данным выявить нельзя, берем самый простой - медиану. PR-AUC у медианы 0,474 против 0,404 и 0,386. Только с медианой Brier лучше постоянного прогноза: масштабированный Brier 0,012 против -0,051 у KNN и -0,068 у MICE. Наклон калибровки у медианы 0,44.
- Проверки. KNN для количественных и медиана для бинарных и порядковых дает 0,675 ± 0,074, почти как медиана везде. Значит, выигрыш дает заполнение бинарных и порядковых признаков. KNN с индикаторами пропусков дает 0,644 ± 0,096, почти как KNN без них.
- Пропуск связан с исходом в нескольких признаках (рисунок 11):
  - тип СД: рецидив 71% среди пропусков против 27% среди заполненных, p < 0,001 (точный критерий Фишера);
  - Невролог: 10% против 34%, p = 0,009 (точный критерий Фишера);
  - ХБП А: 23% против 37%, p = 0,021 (хи-квадрат Пирсона);
  - ретинопатия (17% против 34%, p = 0,054) и ХБП С (23% против 34%, p = 0,093) - без значимости, в ту же сторону.

  У остальных признаков p от 0,176. У Невролога, ХБП А, ХБП С и ретинопатии пропуск чаще у единичных эпизодов, медиана ставит им одно типичное значение. Индикаторы пропусков при этом KNN не помогли, поэтому связь выигрыша медианы с информативным пропуском по этим данным не доказана. Для деревьев на шаге 8 пропуск как сигнал допускаем (решение автора): CatBoost видит пропуск типа СД напрямую.
- Распределения медиана сохраняет хуже всех: медиана KS по признакам 0,495 против 0,237 у KNN и 0,217 у MICE, все пропуски встают в одно значение. Для логрега это допустимо, для описания распределений заполненных данных - нет.
- Выбор импутации сделан на тех же фолдах, что и оценка, поэтому все дальнейшие оценки немного завышены этим выбором.
- MICE за 10 итераций не сходится (ConvergenceWarning в ячейках выше), еще один довод против нее.
- В облаках PCA пациенты с заполненными значениями не образуют отдельного облака ни при одном методе (рисунок 12). Пропусков нет только у 11 пациентов train, у 163 заполнено 4 значения и больше.
- Выраженной структуры в данных нет: для 80% дисперсии нужно 11 главных компонент из 22, первая объясняет 19%. На PCA и t-SNE рецидивы и единичные эпизоды перемешаны.
- k-means: лучший силуэт 0,131 при двух кластерах, это слабое разделение. Доля рецидива в кластерах 35% и 28%, хи-квадрат Пирсона p = 0,275: кластеры с исходом не связаны.

## Шаг 5. Дисбаланс классов

Рецидивов на train 31%. Решение общее для всех моделей, поэтому принимаем его до подбора. Сравниваем три способа обучить бейзлайн-логрег (все признаки, импутация шага 4, L2, C = 1):
- без балансировки;
- веса классов (`class_weight="balanced"`): ошибка на рецидиве весит примерно в 2,2 раза больше (158 единичных к 71 рецидиву на train), данные не меняются;
- SMOTENC: в обучающую часть фолда добавляются синтетические рецидивы до равенства классов. Это единственный вариант SMOTE в imblearn для смешанных данных: количественные признаки интерполируются между соседями, бинарным, порядковым и номинальным достается самое частое значение среди соседей, поэтому недопустимых кодов (ретинопатия 1,4) не возникает. Валидационные пациенты синтетикой не затрагиваются.

Импутация и масштаб одинаковы во всех вариантах. Рабочая точка выбирается порогом (F1 и F2), поэтому сдвиг прогнозов от балансировки сам по себе ничего не дает.

In [ ]:
balance_results = [
    evaluate.cross_validate("логрег, без балансировки", models.logreg(), X_train, y_train),
    evaluate.cross_validate("логрег, веса классов", models.logreg(class_weight="balanced"), X_train, y_train),
    evaluate.cross_validate("логрег, SMOTENC", models.logreg_smote(), X_train, y_train),
]
balance_summary = evaluate.summary_table(balance_results)
save_table(balance_summary.round(3), "step5_imbalance")
balance_summary.round(3).T

In [ ]:
balance_points = pd.concat([evaluate.operating_points(r) for r in balance_results], ignore_index=True)
save_table(balance_points.round(3), "step5_operating_points")
balance_points[balance_points["правило"].isin(evaluate.MAIN_RULES)].round(3).T

In [ ]:
fig = evaluate.plot_calibration(balance_results)
caption(fig, "Калибровка бейзлайн-логрега без балансировки, с весами классов и с SMOTENC: средний прогноз и наблюдаемая доля рецидива в 5 квантильных группах OOF-прогнозов всех 5 повторов. Пунктир - идеальная калибровка.")
save_figure(fig, "step5_calibration")
plt.show()

In [ ]:
fig = evaluate.plot_curves(balance_results)
caption(fig, "ROC- и PR-кривые бейзлайн-логрега при трех способах учесть дисбаланс, OOF-прогнозы на train, все 5 повторов CV вместе.")
save_figure(fig, "step5_curves")
plt.show()

In [ ]:
fig = evaluate.plot_dca(balance_results)
caption(fig, "Кривые принятия решений бейзлайн-логрега при трех способах учесть дисбаланс против стратегий «лечить всех» и «никого».")
save_figure(fig, "step5_dca")
plt.show()
pd.DataFrame([evaluate.useful_thresholds(r) for r in balance_results]).round(2)

### Вывод шага 5

- Бейзлайн-логрег, ROC-AUC OOF: без балансировки 0,681 (95% ДИ 0,614-0,750), веса классов 0,677 (0,609-0,746), SMOTENC 0,651 (0,580-0,717). ДИ перекрываются, лучший вариант выявить нельзя, берем самый простой - без балансировки. PR-AUC 0,474, 0,473 и 0,425.
- Балансировка завышает прогнозы (рисунок 18): калибровка в среднем -0,798 у весов и -0,688 у SMOTENC против -0,024 без балансировки. Масштабированный Brier -0,106 и -0,162, хуже постоянного прогноза, без балансировки 0,012.
- Рабочая точка F1 (порог, помечено, чувствительность, специфичность, PPV, NPV):
  - без балансировки: 0,127; 68%; 0,868; 0,410; 0,400; 0,877;
  - веса классов: 0,329; 61%; 0,803; 0,484; 0,412; 0,845;
  - SMOTENC: 0,099; 73%; 0,896; 0,348; 0,386; 0,903.

  Порог сам сдвигается под сдвиг прогнозов. Точки отличаются соотношением чувствительности и специфичности, PPV у всех 0,39-0,41.
- Кривая принятия решений выше обеих стратегий: без балансировки в основном на 0,16-0,45, веса 0,06-0,41, SMOTENC 0,15-0,40 (рисунок 20).
- Итог: балансировка не нужна, дисбаланс учитывает порог. Для деревьев (шаг 8) это допущение, там тоже без балансировки.

## Шаг 6. Логрег: подбор гиперпараметров

Логрег на всех 24 признаках (импутация шага 4, без балансировки по шагу 5). Штраф L2: он стягивает коэффициенты к нулю, но не обнуляет их, поэтому в модели остаются все признаки. L1 и elastic net обнуляют коэффициенты, это уже отбор, они идут методами шага 7. Сетка: сила регуляризации C, 9 значений от 0,001 до 10 (чем меньше C, тем сильнее штраф). Подбор по log loss во внутренней стратифицированной CV на 5 фолдов внутри каждого внешнего фолда (вложенная CV). Итог шага - модель А: логрег на всех признаках.

Почему log loss, а не ROC-AUC: ROC-AUC зависит только от порядка пациентов по прогнозу, а сила L2-штрафа этот порядок почти не меняет. Тогда значения C почти не различаются по ROC-AUC внутренней CV, и выбор определяет шум. Log loss штрафует и за неверный порядок, и за неверные вероятности, поэтому различает C по калибровке. Проверка в 6.1.

In [ ]:
logreg_all = evaluate.cross_validate("логрег, все признаки", models.logreg_search(), X_train, y_train)
grid_results = [baseline, logreg_all]
grid_summary = evaluate.summary_table(grid_results)
save_table(grid_summary.round(3), "step6_grid")
grid_summary.round(3).T

In [ ]:
# какие параметры выбирались во внешних фолдах: если похожие, подбор устойчив
chosen_params = pd.DataFrame(logreg_all["параметры по фолдам"]).rename(columns=lambda c: c.removeprefix("model__"))
for column in ["C"]:
    counts = chosen_params[column].astype(str).value_counts()
    print(f"{column}: " + ", ".join(f"{value} - {count}" for value, count in counts.items()))
save_table(chosen_params, "step6_params_by_fold")

In [ ]:
search = models.logreg_search().fit(X_train, y_train)
grid_params = selection.model_params(search)
print("параметры на всем train:", grid_params, f"log loss внутренней CV {-search.best_score_:.3f}")

### 6.1 Качество при каждом C

Проверка выбора критерия: логрег на всех признаках с каждым C сетки без подбора, те же 25 фолдов. Это не оценка модели (C здесь не выбирается внутри фолда), а профиль: как меняются различение и калибровка с силой штрафа.

In [ ]:
c_profile = evaluate.summary_table(
    [evaluate.cross_validate(f"C = {c:g}", models.logreg(C=c), X_train, y_train) for c in models.C_GRID]
)
c_columns = ["модель", "ROC-AUC OOF", "ROC-AUC ДИ низ", "ROC-AUC ДИ верх", "Brier масштабированный", "калибровка: наклон"]
save_table(c_profile[c_columns].round(3), "step6_c_profile")
c_profile[c_columns].round(3)

### Вывод шага 6

- C по фолдам выбирается устойчиво: 0,1 в 18 фолдах из 25, 0,032 в 7. На всем train C = 0,032, log loss внутренней CV 0,569.
- Модель А (L2, все 24 признака, C по log loss): ROC-AUC CV 0,685 ± 0,074, OOF 0,678 (95% ДИ 0,606-0,746). Различение как у бейзлайна (0,681, ДИ 0,614-0,750), ДИ почти совпадают.
- Калибровка лучше бейзлайна: наклон 0,797 против 0,442, масштабированный Brier 0,073 против 0,012. Подбор C дал более точные вероятности, а не лучшее ранжирование.
- Профиль по C (6.1): ROC-AUC OOF при всех 9 значениях C от 0,676 до 0,687, ДИ перекрываются. Калибровка при этом меняется сильно: наклон от 10,852 при C = 0,001 (прогнозы сжаты) до 0,349 при C = 10 (прогнозы крайние), масштабированный Brier от 0,090 при C = 0,032 до -0,030 при C = 10. Поэтому C выбираем по log loss: ROC-AUC между значениями C почти не различает. Профиль не вложен, это проверка критерия, а не оценка модели.
- Итог шага - модель А: логрег на всех 24 признаках, L2, C = 0,032.

## Шаг 7. Логрег: отбор признаков

### 7.1 Одномерная разделяющая сила признаков

Описание, в модель напрямую не идет. Каждый признак оценен на своих непропущенных значениях, номинальные без AUC. Исключенные из модели возраст манифестации и ЛПНП показаны отдельными строками, чтобы было видно, могли ли они попасть в топ (почему они исключены - в 0.6 и 0.7).

In [ ]:
excluded = [config.MANIFESTATION, config.LDL]
X_screen = X_train.join(labeled.set_index(config.ID_COLUMN)[excluded])
screen = screening.screen(X_screen, y_train, excluded=excluded)
save_table(screen, "step7_screening")
fig = screening.plot_forest(screen)
caption(fig, f"Одномерный ROC-AUC признаков на train ({len(y_train)} пациентов) с 95% бутстрэп-ДИ, включая исключенные из модели возраст манифестации и ЛПНП. Защитные признаки развернуты, чтобы AUC был не ниже 0,5. Пунктир - случайное угадывание.")
save_figure(fig, "step7_screening")
plt.show()
screen

### 7.2 Пять методов отбора и голосование

Параметры логрега берем из сетки шага 6 и с ними отбираем признаки пятью методами (`src/selection.py`). Отбираются исходные признаки: номинальный признак входит или выходит всеми уровнями. Качество набора - ROC-AUC внутренней CV.
- одномерный: признаки упорядочены по ROC-AUC логрега на одном признаке и добавляются по этому порядку;
- жадный вперед: на каждом шаге добавляется признак, с которым ROC-AUC выше всего;
- жадный назад: от всех признаков на каждом шаге убирается признак, без которого ROC-AUC выше всего;
- L1: логрег с L1-штрафом на всех признаках, C подбирается внутренней CV, отобраны признаки с ненулевыми коэффициентами;
- elastic net: то же, подбираются C и доля L1.

Остановка у первых трех методов: если 2 шага подряд ROC-AUC не превысил лучший, поиск заканчивается и берется набор с лучшим значением. Итоговый набор (голосование) - признаки, которые отобрали не меньше 3 методов из 5. Итог шага - модель Б: логрег на наборе голосования.

Все вложено: в каждом внешнем фолде сетка, отбор и голосование делаются заново на обучающей части, модель проверяется на валидационной.

In [ ]:
method_results = selection.cross_validate_methods(X_train, y_train)
logreg_vote = method_results[-1]
selection_results = [logreg_all] + method_results
selection_summary = evaluate.summary_table(selection_results)
save_table(selection_summary.round(3), "step7_methods")
selection_summary.round(3).T

In [ ]:
fig = evaluate.plot_cv_auc(selection_summary)
caption(fig, "ROC-AUC OOF с 95% бутстрэп-ДИ: логрег на всех признаках, на наборах пяти методов отбора и на наборе голосования. Отбор и подбор параметров внутри каждого внешнего фолда.")
save_figure(fig, "step7_cv_auc")
plt.show()

In [ ]:
method_frequency = selection.selection_frequency(method_results, list(X_train.columns))
method_frequency = method_frequency.loc[method_frequency.mean(axis=1).sort_values(ascending=False).index]
save_table(method_frequency.round(2).reset_index(names="признак"), "step7_selection_frequency")
fig = screening.plot_selection_frequency(method_frequency.rename(index=eda.short_name))
caption(fig, "Доля из 25 внешних фолдов, в которых признак отобран каждым методом и голосованием. Отбор на обучающей части фолда.")
save_figure(fig, "step7_selection_frequency")
plt.show()

### 7.3 Отбор на всем train

Те же методы на всем train: наборы признаков модели Б.

In [ ]:
vote_full = selection.SelectedLogreg().fit(X_train, y_train)
sets_table = selection.agreement(vote_full.sets_, list(X_train.columns))
save_table(sets_table.reset_index(names="признак"), "step7_sets")
print("набор голосования:", [eda.short_name(f) for f in vote_full.features_])
sets_table.rename(index=eda.short_name)

### Вывод шага 7

- Одномерно (рисунок 21) сильнее всего отделяют рецидив суточная доза (ROC-AUC 0,697, ДИ 0,617-0,767), HbA1c (0,678), ЛПВП (0,643) и ХБП С (0,629). Нижняя граница ДИ выше 0,5 еще у длительности СД, ретинопатии и Невролога. Исключенные признаки в топ бы не попали: возраст манифестации 0,557 (0,476-0,636), ЛПНП 0,519 (0,410-0,625).
- Вложенная оценка методов, ROC-AUC OOF (рисунок 22): одномерный 0,713 (0,646-0,774), жадный вперед 0,693, жадный назад 0,687, L1 0,689, elastic net 0,682, голосование 0,700 (0,633-0,764), все признаки 0,678. ДИ перекрываются: по различению отбор модель не улучшает и не ухудшает. Калибровка у голосования близка к идеальной: наклон 0,964, масштабированный Brier 0,098.
- Устойчивость по 25 фолдам (рисунок 23), доля фолдов в наборе голосования: вид инсулинотерапии и суточная доза 100%, ретинопатия и ХБП С 96%, HbA1c 88%, калий 80%, ЛПВП 76%, натрий 68%, остальные 40% и меньше. Глюкоза - 36%.
- На всем train (7.3) все 5 методов отобрали 7 признаков: суточную дозу, вид инсулинотерапии, ретинопатию, HbA1c, ЛПВП, калий и глюкозу. ХБП С отобрали 3 метода (кроме жадных). L1 и elastic net взяли еще мочевину, BE, натрий и Невролог, одномерный - длительность СД и ХБП А.
- Итог шага - модель Б: логрег на 8 признаках голосования (7 признаков выше и ХБП С), L2, C = 0,032. Глюкоза вошла в набор на всем train, но по фолдам неустойчива.

## Шаг 8. Бустинги и случайный лес

Три модели, каждая подбирает параметры Optuna во вложенной CV (`src/tuning.py`): внутри каждого из 25 внешних фолдов 40 проб (10 случайных, дальше TPE) по ROC-AUC внутренней стратифицированной CV на 5 фолдов, затем обучение с лучшими параметрами на всей обучающей части фолда. Балансировки нет (шаг 5).

- CatBoost видит пропуски напрямую: числовые NaN обрабатывает сам (параметр nan_mode), у номинальных пропуск - отдельный уровень "пропуск" (пропуск как сигнал допускаем).
- Случайный лес и LightGBM учатся на импутации шага 4: медиана для числовых, мода и OHE для номинальных. Число деревьев леса не подбираем, 500.

Пространства заданы заново на нынешнем train, широкими. После прогона проверяем, сошелся ли подбор и не упирается ли выбор в края диапазонов (8.1). Подбор идет около часа, поэтому результаты сохраняются в `data/interim/step8` и при повторном прогоне читаются с диска, если совпадают данные (отпечаток значений признаков и исхода), фолды, зерно, модели, пространства и число проб. У LightGBM число листьев закреплено на 64, сложность дерева задает глубина.

Признаки деревьям не отбираем, сложность ограничивают параметры Optuna. Встроенные меры важности (gain у LightGBM, MDI у леса) для отбора не годятся: они считаются на обучающих данных и завышают признаки с множеством разных значений, даже чистый шум (Strobl et al., BMC Bioinformatics, 2007, 8:25).

In [ ]:
tuning.space_table()

In [ ]:
tree_results = [tuning.cross_validate_cached(name, X_train, y_train) for name in tuning.SPACES]
tree_summary = evaluate.summary_table(tree_results)
save_table(tree_summary.round(3), "step8_models")
tree_summary.round(3).T

### 8.1 Проверка подбора

Сошелся ли подбор за 40 проб и не упирается ли выбор параметров в края диапазонов. Если лучший результат растет до последних проб, проб мало. Если во многих фолдах выбор у края, диапазон узок.

In [ ]:
tree_colors = evaluate.model_colors([logreg_all, logreg_vote] + tree_results)[2:]  # цвета как на сравнительных графиках шага 9
fig = tuning.plot_convergence(tree_results, tree_colors)
caption(fig, "Лучший ROC-AUC внутренней CV по ходу 40 проб Optuna в каждом из 25 внешних фолдов (тонкие линии) и медиана по фолдам (черная линия).")
save_figure(fig, "step8_convergence")
plt.show()
convergence = tuning.convergence_table(tree_results)
save_table(convergence.round(4), "step8_convergence")
convergence.round(4)

In [ ]:
fig = tuning.plot_chosen(tree_results, tree_colors)
caption(fig, f"Положение выбранных значений числовых параметров в их диапазонах по 25 внешним фолдам: 0 - нижний край, 1 - верхний. Серые полосы - {tuning.EDGE:.0%} диапазона у краев.")
save_figure(fig, "step8_chosen_params")
plt.show()
boundaries = pd.concat([tuning.boundary_table(r) for r in tree_results], ignore_index=True)
save_table(boundaries.round(3), "step8_boundaries")
boundaries.round(3)

In [ ]:
choices = pd.concat([tuning.choice_counts(r) for r in tree_results], ignore_index=True)
save_table(choices, "step8_choices")
choices

In [ ]:
TREES = ["CatBoost", "случайный лес"]  # LightGBM дальше шага 8 не идет (вывод шага 8)
trees = [r for r in tree_results if r["модель"] in TREES]

### Вывод шага 8

- ROC-AUC OOF: CatBoost 0,681 (95% ДИ 0,621-0,743), случайный лес 0,685 (0,622-0,746), LightGBM 0,661 (0,599-0,716). Средний ROC-AUC по фолдам 0,707, 0,698 и 0,687. ДИ перекрываются между собой и с бейзлайном (0,681).
- Подбор сошелся (рисунок 24): у CatBoost и леса медиана лучшего ROC-AUC внутренней CV после пробы 30 не растет (0,7610 и 0,7322). У LightGBM рост после пробы 30 есть в 15 фолдах из 25, но медиана поднялась только с 0,7298 до 0,7319.
- Края диапазонов (рисунок 25): чаще всего выбор упирается в естественные нижние границы. Глубина CatBoost 1 в 40% фолдов (медиана 2), минимальный лист леса у нижнего края в 40% фолдов, глубина LightGBM в 36%. Глубина 1 значит, что CatBoost складывает пни: каждое дерево делит пациентов по одному признаку, модель аддитивная, без взаимодействий признаков.
- Калибровка: наклон CatBoost 1,339 и леса 1,272 (прогнозы сжаты к средней доле), LightGBM 0,497 (прогнозы слишком крайние). Масштабированный Brier 0,079, 0,098 и 0,018.
- LightGBM дальше не берем: худший ROC-AUC OOF из трех, переобучен (наклон 0,497) и по Brier почти не лучше постоянного прогноза. В сравнение и интерпретацию идут CatBoost и случайный лес на всех 24 признаках.

## Шаг 9. Сравнение моделей и выбор

Модели с вложенной оценкой на одних и тех же фолдах: логрег на всех признаках (А), логрег на наборе голосования (Б), CatBoost и случайный лес на всех признаках. Если ДИ ROC-AUC перекрываются, лучшую модель выявить нельзя, и при равенстве берем более простую. Финальные модели (для статьи и для сервиса) выбирает автор по этим таблицам.

In [ ]:
main_results = [logreg_all, logreg_vote] + trees
comparison = evaluate.summary_table(main_results)
save_table(comparison.round(3), "step9_models")
fig = evaluate.plot_cv_auc(comparison)
caption(fig, "ROC-AUC OOF с 95% бутстрэп-ДИ: логрег на всех признаках (А) и на наборе голосования (Б), CatBoost и случайный лес. Все оценки вложенные.")
save_figure(fig, "step9_cv_auc")
plt.show()
comparison.round(3).T

In [ ]:
comparison_points = pd.concat([evaluate.operating_points(r) for r in main_results], ignore_index=True)
save_table(comparison_points.round(3), "step9_operating_points")
comparison_points[comparison_points["правило"].isin(evaluate.MAIN_RULES)].round(3).T

In [ ]:
fig = evaluate.plot_curves(main_results)
caption(fig, "ROC- и PR-кривые логрегов А и Б, CatBoost и случайного леса, OOF-прогнозы на train, все 5 повторов CV вместе.")
save_figure(fig, "step9_curves")
plt.show()

In [ ]:
fig = evaluate.plot_calibration(main_results)
caption(fig, "Калибровка логрегов А и Б, CatBoost и случайного леса: средний прогноз и наблюдаемая доля рецидива в 5 квантильных группах OOF-прогнозов всех 5 повторов. Пунктир - идеальная калибровка.")
save_figure(fig, "step9_calibration")
plt.show()

In [ ]:
fig = evaluate.plot_dca(main_results)
caption(fig, "Кривые принятия решений логрегов А и Б, CatBoost и случайного леса против стратегий «пометить всех» и «никого».")
save_figure(fig, "step9_dca")
plt.show()
pd.DataFrame([evaluate.useful_thresholds(r) for r in main_results])

### 9.1 Финальные конфигурации и пороги

Вложенная оценка описывает процедуру, где в каждом фолде свои параметры (и у логрега Б свой набор). На тест и в сервис пойдет одна модель, обученная на всем train. Ее прогнозы лежат в другой шкале, чем смесь моделей фолдов, поэтому калибровку и порог смотрим по OOF финальной конфигурации: те же 25 фолдов, параметры (и набор) выбраны на всем train и фиксированы. Эта оценка не вложена, для различения основной остается вложенная.

Порог: медиана порогов максимума F1 (и отдельно F2) по 5 повторам OOF финальной конфигурации. На тесте порог будет один, поэтому рядом метрики при одном фиксированном пороге во всех повторах. Модели замораживаются после выбора автора.

In [ ]:
final_models = {"логрег А": search.best_estimator_, "логрег Б": vote_full.model_}  # обучены на всем train
final_builds = {
    "логрег А": models.logreg(**grid_params),
    "логрег Б": models.logreg(vote_full.features_, **vote_full.params_),
}
for name in TREES:
    found = tuning.search(name).fit(X_train, y_train)
    final_models[name] = found.best_estimator_
    final_builds[name] = tuning.BUILDERS[name](**found.best_params_)
    print(f"{name}: ROC-AUC внутренней CV {found.best_score_:.3f}, параметры {found.best_params_}")

In [ ]:
final_configs = [
    evaluate.cross_validate(f"{name}, финальная конфигурация", build, X_train, y_train)
    for name, build in final_builds.items()
]
final_summary = evaluate.summary_table(final_configs)
save_table(final_summary.round(3), "step9_final_models")
final_summary.round(3).T

In [ ]:
final_points, fixed_points = [], []
for result in final_configs:
    points = evaluate.operating_points(result)
    rule_thresholds = points.set_index("правило")["порог"]
    final_points.append(points)
    fixed_points.append(evaluate.fixed_threshold_points(result, {rule: rule_thresholds[rule] for rule in evaluate.MAIN_RULES}))
final_points = pd.concat(final_points, ignore_index=True)
fixed_points = pd.concat(fixed_points, ignore_index=True)
save_table(final_points.round(3), "step9_final_operating_points")
save_table(fixed_points.round(3), "step9_final_fixed_threshold")
fixed_points.round(3).T

In [ ]:
fig = evaluate.plot_calibration(final_configs)
caption(fig, "Калибровка финальных конфигураций (параметры и набор с всего train, одни во всех фолдах): средний прогноз и наблюдаемая доля рецидива в 5 квантильных группах OOF-прогнозов всех 5 повторов. Пунктир - идеальная калибровка.")
save_figure(fig, "step9_final_calibration")
plt.show()

### Вывод шага 9

- Вложенная оценка, ROC-AUC OOF (рисунок 26): логрег А 0,678 (95% ДИ 0,606-0,746), логрег Б 0,700 (0,633-0,764), CatBoost 0,681 (0,621-0,743), случайный лес 0,685 (0,622-0,746). ДИ всех четырех моделей перекрываются между собой и с бейзлайном (0,681): лучшую модель по этим данным выявить нельзя. При равенстве проще логрег.
- Калибровка (рисунок 28): наклон А 0,797, Б 0,964, CatBoost 1,339, лес 1,272 (у деревьев прогнозы сжаты к средней доле). Масштабированный Brier 0,073, 0,098, 0,079 и 0,098.
- Рабочая точка F1 по вложенной оценке (порог, помечено, чувствительность, специфичность, PPV, NPV):
  - логрег А: 0,214; 66%; 0,856; 0,433; 0,408; 0,886;
  - логрег Б: 0,223; 67%; 0,887; 0,425; 0,411; 0,901;
  - CatBoost: 0,279; 68%; 0,899; 0,415; 0,410; 0,911;
  - случайный лес: 0,264; 71%; 0,927; 0,392; 0,407; 0,925.

  PPV у всех около 0,41: рецидив примерно у 2 из 5 помеченных.
- Кривые принятия решений всех четырех моделей выше обеих стратегий примерно от 0,05 до 0,45-0,52 (рисунок 29).
- Финальные конфигурации (9.1, не вложено): А 0,685 (0,615-0,752), Б 0,755 (0,688-0,816), CatBoost 0,721 (0,656-0,781), лес 0,708 (0,644-0,769). У Б оценка на 0,055 выше вложенной: набор выбран на том же train, поэтому она оптимистична. Для различения моделей опираемся на вложенную оценку.
- Калибровка финальных конфигураций (рисунок 30): наклон А 1,145, Б 1,749, CatBoost 1,630, лес 1,436, прогнозы сжаты к средней доле. Калибровка в среднем от -0,046 до -0,002.
- Пороги финальных конфигураций (медиана по 5 повторам), F1 и F2: А 0,212 и 0,185; Б 0,287 и 0,219; CatBoost 0,304 и 0,206; лес 0,270 и 0,227.
- При одном фиксированном пороге F1 (помечено, чувствительность, специфичность, PPV, NPV):
  - А: 77%; 0,946; 0,305; 0,380; 0,927;
  - Б: 56%; 0,806; 0,544; 0,443; 0,862;
  - CatBoost: 60%; 0,856; 0,519; 0,445; 0,890;
  - лес: 69%; 0,927; 0,414; 0,415; 0,927.
- Какие модели идут в статью и в сервис, решает автор. Тест открываем после этого решения.

## Шаг 10. Интерпретация: топ-факторы и совместные эффекты

Главный вопрос работы: какие признаки связаны с рецидивом и как. Все на train, тест закрыт. SHAP описывает, на что опирается модель, а не причину рецидива. Размеры эффекта отдельных признаков дает статистика шага 1, ОШ из модели не считаем (решение автора).

### 10.1 Рейтинг по SHAP

Модели обучены на всем train (9.1). Основной рейтинг - по среднему модулю SHAP, одна мера на модель. Деревья сводим вместе медианным рангом, логреги показываем отдельно: логрег Б берет только свой набор, у остальных признаков у него последний ранг. Встроенные меры деревьев справочно: MDI у леса завышает признаки с множеством разных значений. Уровни номинального признака сложены в один исходный признак.

In [ ]:
features = list(X_train.columns)
shap_tables = {name: importance.shap_values(name, fitted, X_train, X_train) for name, fitted in final_models.items()}
shap_measures = {name: importance.shap_by_feature(values, features) for name, values in shap_tables.items()}
ranks = importance.rank_table({name: shap_measures[name] for name in TREES})
for name in ["логрег А", "логрег Б"]:
    ranks[name] = importance.rank_table({name: shap_measures[name]})[name]
save_table(ranks.reset_index(names="признак"), "step10_ranks")
fig = importance.plot_ranks(ranks, "Ранг признака по SHAP (1 - самый важный)")
caption(fig, "Ранги 12 признаков с наименьшим медианным рангом по среднему модулю SHAP CatBoost и случайного леса, обученных на всем train; логреги справа, в медиану не входят. 1 - самый важный, признак без вклада в модель получает последний ранг 24.")
save_figure(fig, "step10_ranks")
plt.show()
ranks.round(1)

In [ ]:
native = {}
for name in TREES:
    for kind, values in importance.native_importance(name, final_models[name], X_train).items():
        native[f"{kind}, {name}"] = values
native_ranks = importance.rank_table(native)
save_table(native_ranks.reset_index(names="признак"), "step10_native_ranks")
fig = importance.plot_ranks(native_ranks, "Ранг признака по встроенным мерам деревьев")
caption(fig, "Справочно: ранги по встроенным мерам важности деревьев, обученных на всем train (PredictionValuesChange у CatBoost, MDI у леса). Эти меры завышают признаки с множеством разных значений.")
save_figure(fig, "step10_native_ranks")
plt.show()

In [ ]:
for name, file in [("логрег А", "step10_shap_logreg_all"), ("логрег Б", "step10_shap_logreg_vote"), ("CatBoost", "step10_shap_catboost")]:
    fig = importance.plot_shap_beeswarm(importance.sum_levels(shap_tables[name]), X_train, f"SHAP, {name}")
    caption(fig, f"SHAP модели {name} на train: точка - пациент, по горизонтали - вклад признака в прогноз (логит), вправо - к рецидиву. Цвет - значение признака от минимума (синий) до максимума (красный), светло-серый - пропуск, темно-серый - известный уровень номинального признака.")
    save_figure(fig, file)
    plt.show()

In [ ]:
# идет ли вклад признака у CatBoost от самого пропуска
missing_shap = importance.shap_by_missing(
    importance.sum_levels(shap_tables["CatBoost"]), X_train, ["тип СД (1-1, 2-2, 3 -др)", "Невролог"]
)
save_table(missing_shap.round(3), "step10_shap_missing")
missing_shap.round(3)

In [ ]:
# лидеры SHAP у пациентов без инсулинотерапии (вид 0) и у остальных
therapy = X_train["Вид инсулинотерапии (1 - ручки, 2 - помпа) на момент ДКА"]
no_therapy = therapy == 0
leaders = pd.DataFrame(
    {
        group: {
            "пациентов": int(mask.sum()),
            "рецидивов": int(y_train[mask].sum()),
            "доля с суточной дозой 0": (X_train.loc[mask, "Суточная доза инсулина"] == 0).mean(),
            "доля с длительностью СД 0": (X_train.loc[mask, config.DURATION] == 0).mean(),
        }
        for group, mask in [("вид 0", no_therapy), ("вид 1 или 2", therapy.isin([1, 2])), ("вид неизвестен", therapy.isna())]
    }
)
save_table(leaders.round(3).reset_index(names="показатель"), "step10_no_therapy")
leaders.round(3)

### 10.2 Устойчивость топа

Бутстрэп: 200 ресэмплов train, модель обучается заново с теми же параметрами, SHAP считается на исходном train. Для сравнения 25 внешних фолдов: в каждом модель с параметрами этого фолда (у логрега Б и свой набор) обучается на обучающей части, SHAP считается на валидационной, то есть на невиденных пациентах.

In [ ]:
frequency = pd.DataFrame(
    {name: importance.top_frequency(name, build, X_train, y_train) for name, build in final_builds.items()}
)
frequency = frequency.loc[frequency.mean(axis=1).sort_values(ascending=False).index]
save_table(frequency.round(3).reset_index(names="признак"), "step10_top_frequency")
fig = importance.plot_top_frequency(frequency, f"Доля бутстрэп-ресэмплов, где признак в топ-{importance.TOP_SIZE} по SHAP")
caption(fig, "Доля из 200 бутстрэп-ресэмплов train, в которых признак вошел в топ-5 по среднему модулю SHAP. Модель обучается заново на каждом ресэмпле с теми же параметрами.")
save_figure(fig, "step10_top_frequency")
plt.show()
frequency.round(2)

In [ ]:
all_features_params = [
    {"признаки": features, **{key.removeprefix("model__"): value for key, value in params.items()}}
    for params in logreg_all["параметры по фолдам"]
]
fold_params = {"логрег А": all_features_params, "логрег Б": logreg_vote["параметры по фолдам"]}
fold_params.update({r["модель"]: r["параметры по фолдам"] for r in trees})
fold_tables = {name: importance.fold_shap(name, params, X_train, y_train) for name, params in fold_params.items()}
fold_ranks = importance.rank_table({name: fold_tables[name].mean() for name in TREES})
for name in ["логрег А", "логрег Б"]:
    fold_ranks[name] = importance.rank_table({name: fold_tables[name].mean()})[name]
fold_frequency = pd.DataFrame(
    {name: table.apply(importance.in_top, axis=1).mean() for name, table in fold_tables.items()}
)
fold_frequency = fold_frequency.loc[fold_frequency.mean(axis=1).sort_values(ascending=False).index]
save_table(fold_ranks.reset_index(names="признак"), "step10_fold_ranks")
save_table(fold_frequency.round(3).reset_index(names="признак"), "step10_fold_top_frequency")
fig = importance.plot_top_frequency(fold_frequency, f"Доля внешних фолдов, где признак в топ-{importance.TOP_SIZE} по SHAP")
caption(fig, "Доля из 25 внешних фолдов, в которых признак вошел в топ-5 по среднему модулю SHAP на валидационной части фолда. В каждом фолде модель с параметрами, выбранными в этом фолде.")
save_figure(fig, "step10_fold_top_frequency")
plt.show()
fold_ranks.round(1)

### 10.3 Совместные эффекты

Взаимодействия SHAP (`src/interactions.py`): вклад каждого признака в прогноз пациента раскладывается на вклад самого признака и вклады пар с другими. Средний модуль вклада пары показывает, какие признаки действуют вместе сильнее суммы по одному. Считаем у случайного леса: CatBoost на всем train - пни (глубина 1, 9.1), взаимодействий у него нет по построению. Вклады леса в единицах вероятности. При 71 рецидиве совместные эффекты - гипотезы для врачей.

In [ ]:
values = interactions.interaction_values("случайный лес", final_models["случайный лес"], X_train)
pairs = interactions.top_pairs(values, features)
pairs.insert(0, "место", range(1, len(pairs) + 1))
save_table(pairs.round(4), "step10_pairs")
pairs.assign(**{c: pairs[c].map(eda.short_name) for c in ["признак 1", "признак 2"]}).round(4)

In [ ]:
first, second = pairs.loc[0, ["признак 1", "признак 2"]]
fig = interactions.plot_pair(importance.sum_levels(shap_tables["случайный лес"]), X_train, first, second, "случайный лес: самая сильная пара")
caption(fig, f"Вклад SHAP признака «{eda.short_name(first)}» в прогноз случайного леса против его значения, цвет - значение признака «{eda.short_name(second)}». Если при одном значении первого признака точки разного цвета расходятся по вертикали, эффект первого зависит от второго.")
save_figure(fig, "step10_pair_forest")
plt.show()

### 10.4 Правила для врачей

Дерево решений на всех признаках. Каждый лист читается как «если ... и ..., то рецидив у такой-то доли пациентов». Признаки идут как есть, без заполнения: пороги в единицах признака, куда идет пропуск, написано в правиле. Номинальные признаки идут кодом.

Параметры дерева подбираются сеткой во вложенной CV, как у логрега (`interactions.RULE_GRID`): глубина 2-5 (глубже правило не прочитать), минимум пациентов в листе 5-30 (меньше 5 - доля рецидива держится на 2-3 людях), обрезка слабых веток и критерий разбиения. Критерий подбора - Brier: доли рецидива в листах должны быть точными. Log loss не годится, у листа с долей 0 или 1 он бесконечен при одном пациенте с другим исходом. Для сравнения рядом исходный вариант, заданный вручную (глубина 3, лист 10). Калибровочный наклон у дерева не показываем: листья с долей 0 или 1 делают его бессмысленным, точность вероятностей видна по Brier. Устойчивость правил - бутстрэпом, в скольких ресэмплах признак входит в дерево.

In [ ]:
fixed_name = f"дерево-правила, глубина {interactions.RULE_DEPTH}, лист {interactions.RULE_LEAF}"
rules_results = [
    evaluate.cross_validate(fixed_name, interactions.rules_model(max_depth=interactions.RULE_DEPTH, min_samples_leaf=interactions.RULE_LEAF), X_train, y_train),
    evaluate.cross_validate("дерево-правила, подбор", interactions.rules_search(), X_train, y_train),
]
rules_summary = evaluate.summary_table(rules_results)
rules_columns = [c for c in rules_summary.columns if c.startswith(("модель", "ROC-AUC", "PR-AUC"))] + ["Brier", "Brier масштабированный"]
rules_summary = rules_summary[rules_columns]
save_table(rules_summary.round(3), "step10_rules_quality")
rules_summary.round(3).T

In [ ]:
# что выбирала сетка во внешних фолдах
rules_by_fold = pd.DataFrame(rules_results[1]["параметры по фолдам"])
for column in rules_by_fold.columns:
    counts = rules_by_fold[column].astype(str).value_counts()
    print(f"{column}: " + ", ".join(f"{value} - {count}" for value, count in counts.items()))
save_table(rules_by_fold, "step10_rules_params_by_fold")

In [ ]:
rules_found = interactions.rules_search().fit(X_train, y_train)
rules_params = rules_found.best_params_
print("параметры на всем train:", rules_params, f"Brier внутренней CV {-rules_found.best_score_:.3f}")
rules_tree = rules_found.best_estimator_
rules = interactions.rules_table(rules_tree, X_train, y_train)
save_table(rules.round(3), "step10_rules")
rules.round(3)

In [ ]:
rules_stability = interactions.rules_stability(X_train, y_train, rules_params)
save_table(rules_stability.round(3).reset_index(name="доля ресэмплов").rename(columns={"index": "признак"}), "step10_rules_stability")
rules_stability.rename(index=eda.short_name).round(2)

### 10.5 Клинические пороги и сочетания признаков

Разведочный статистический анализ на всех 287 пациентах с исходом, как шаг 1 (`src/thresholds.py`). Для факторов ядра и второго ряда порог выбирается по максимуму индекса Юдена, направление - по данным (выше значение - чаще рецидив или наоборот). Для порога: чувствительность, специфичность, доля рецидива при выполненном условии и без него, ОШ и p по протоколу. Порог найден на тех же пациентах, поэтому метрики при нем оптимистичны. Насколько порог устойчив, показывает 95% ДИ бутстрэпом (1000 ресэмплов).

Сочетания: заранее заданные 4 пары с суточной дозой, первым фактором всех моделей. Оба условия выполнены против остальных, среди пациентов, у которых известны оба признака. Пары не перебираются: при сыром p < 0,05 перебор нашел бы случайные значимые сочетания.

In [ ]:
cutoff_table = thresholds.threshold_table(X, y)
save_table(cutoff_table.round(3), "step10_cutoffs")
cutoff_table.drop(columns=["признак", "направление"]).round(3)

In [ ]:
pair_cutoffs = thresholds.pair_table(X, y, cutoff_table)
save_table(pair_cutoffs.round(3), "step10_cutoff_pairs")
pair_cutoffs.round(3)

### 10.6 Сводная таблица топ-факторов

Все взгляды на признак в одной таблице: сколько из 5 методов отбора логрега взяли его на всем train и вошел ли он в набор голосования (7.3), в какой доле внешних фолдов он попадал в набор голосования (7.2), ранг по SHAP у логрега А и медианный ранг у CatBoost и леса (10.1). Ранг 1 - самый важный, 24 - признак без вклада.

In [ ]:
summary = pd.DataFrame(
    {
        "методов отбора из 5": sets_table["голосов"],
        "в наборе голосования": sets_table[selection.VOTE],
        "доля фолдов в голосовании": method_frequency[selection.VOTE],
        "ранг SHAP, логрег А": ranks["логрег А"],
        "ранг SHAP, деревья": ranks["медианный ранг"],
    }
).sort_values(["методов отбора из 5", "ранг SHAP, деревья"], ascending=[False, True])
save_table(summary.round(2).reset_index(names="признак"), "step10_summary")
summary.rename(index=eda.short_name).round(2)

### Вывод шага 10

- Суточная доза инсулина - первый фактор во всех взглядах: ранг 1 по SHAP у всех четырех моделей (рисунок 31), в топ-5 в 95,5-99,5% бутстрэп-ресэмплов (рисунок 36) и в 92-100% внешних фолдов (рисунок 37), отобрана всеми 5 методами. На ней же первое разбиение дерева правил в 83% ресэмплов.
- Вид инсулинотерапии: отобран всеми 5 методами, в наборе голосования в 100% фолдов, в топ-5 по фолдам у всех моделей (80-100%). В бутстрэпе у CatBoost в топ-5 только в 13,5% ресэмплов. Признак во многом выделяет пациентов без инсулинотерапии (вид 0): 48 пациентов train, 1 рецидив, доза 0 у 75%, длительность СД 0 у 79%.
- ХБП С и ретинопатия: у логрегов на 2-3 месте по SHAP и в топ-5 в 92-100% фолдов, в наборе голосования в 96% фолдов. У деревьев ХБП С на 4 месте, ретинопатия на 9, в топ-5 по фолдам 16-48%.
- Длительность СД важна только деревьям: ранг 2-3, в топ-5 в 84% фолдов у обоих. У логрега А ранг 13, в топ-5 ни в одном фолде, в набор голосования не входит. Деревья и логрег используют признак по-разному, причину по этим данным не проверяли.
- ЛПВП, глюкоза, HbA1c и калий отобраны всеми 5 методами на всем train, но по фолдам в топ-5 SHAP входят редко (4-48%). HbA1c у CatBoost вклада не имеет (ранг 24), у леса на 5 месте. Это факторы второго ряда, их вклад зависит от модели.
- Тип СД важен только CatBoost (ранг 3) и только через пропуск: средний SHAP 0,364 у 21 пациента с пропуском против -0,030 у остальных.
- Совместные эффекты (10.3, рисунок 38): самая сильная пара у леса - суточная доза и ХБП С, средний модуль вклада 0,0065 (0,65 процентного пункта вероятности). У остальных пар 0,0031 и меньше. Совместные эффекты малы. Это согласуется с выбором глубины 1 у CatBoost: взаимодействия не окупаются.
- Правила (10.4): подбор сетки выбирает простое дерево. Во внешних фолдах глубина 2 в 18 из 25, минимальный лист 15-30 в 19. На всем train: глубина 2, лист 15, обрезка 0,005. Три правила:
  - «доза > 31,5 или пропуск, СД 3 типа или тип не указан»: 26 пациентов, 18 рецидивов (69%);
  - «доза > 31,5 или пропуск, тип СД 1 или 2»: 134 пациента, 49 рецидивов (37%);
  - «доза ≤ 31,5»: 69 пациентов, 4 рецидива (6%).

  Вложенно ROC-AUC OOF 0,637 (0,582-0,688) против 0,617 у дерева глубины 3 с листом 10, ДИ перекрываются. Масштабированный Brier -0,017 против -0,091: подбор сделал доли в листах точнее, но они все еще не точнее постоянного прогноза. Устойчиво только разбиение по дозе (80,5% ресэмплов), тип СД входит в дерево в 22,5%. Первое правило держится на пропуске типа СД (4.2: при пропуске рецидив у 71% на train). Данные поддерживают правила не длиннее двух условий, это согласуется с глубиной 1 у CatBoost и слабыми взаимодействиями. Правила - гипотезы для врачей, не готовые критерии.
- Клинические пороги (10.5, все 287 пациентов, разведочно). Порог дозы по Юдену 33 (95% ДИ бутстрэпом 22-38), рядом с разбиением дерева правил (31,5): при дозе от 33 рецидив у 70 из 160 (44%), ниже у 13 из 104 (12,5%), чувствительность 0,843, специфичность 0,503, ОШ 5,44 (2,82-10,53). HbA1c ≤ 9,2: чувствительность 0,333, специфичность 0,934, рецидив у 13 из 18 против 26 из 97, но ДИ порога широкий (8,92-13,04). ЛПВП ≥ 1,03, ХБП С от 1 и ретинопатия от 1 дают ОШ 2,36-3,09. Длительность больше 0 отделяет дебют: среди 56 дебютов рецидивов нет по определению исхода, это не фактор риска.
- Сочетания с дозой ≥ 33: с HbA1c ≤ 9,2 рецидив у 7 из 8 против 29 из 99 (ОШ 16,9, ДИ 1,99-143,5, точный критерий Фишера p = 0,002), но сочетание ловит только 7 из 36 рецидивов (чувствительность 0,194, специфичность 0,986) и держится на 8 пациентах. С ХБП С от 1: 32 из 55 против 29 из 124 (ОШ 4,56), с ЛПВП ≥ 1,03: 30 из 55 против 21 из 92 (ОШ 4,06). Пороги найдены на тех же данных, числа оптимистичны, это гипотезы для проверки.
- Сводная таблица (10.6): надежное ядро, которое отбирают почти все методы и которое устойчиво по фолдам, - суточная доза, вид инсулинотерапии, ХБП С и ретинопатия. Длительность СД важна деревьям. ЛПВП, HbA1c, калий и глюкоза - второй ряд.

## Шаг 11. Проверка на тесте

Модели сервиса выбрал автор (2026-09-27): логрег Б и CatBoost в финальной конфигурации (9.1), обученные на всем train. Логрег А остается для анализа SHAP (шаг 10) и на тесте не проверяется. Пороги - медиана порогов максимума F1 и F2 по 5 повторам OOF финальной конфигурации (9.1).

Порядок зафиксирован до открытия теста:
- модели и пороги замораживаются до теста (`src/finalist.py`): параметры, коэффициенты логрега, контрольная сумма прогнозов CatBoost на train, отпечаток данных. При повторном прогоне запись сверяется, и если что-то изменилось, ноутбук падает;
- на 58 пациентах теста (18 рецидивов): ROC-AUC и PR-AUC с бутстрэп-ДИ, Brier с эталоном, калибровка, матрица ошибок при замороженных порогах, DCA. Порог на тесте не подбирается;
- согласие с валидацией: попадает ли ROC-AUC теста в 95% ДИ OOF финальной конфигурации, и метрики при пороге рядом с OOF при том же фиксированном пороге (9.1);
- результат отчитываем как есть, модели и пороги после теста не меняются.

In [ ]:
SERVICE = ["логрег Б", "CatBoost"]  # выбор автора 2026-09-27
service_thresholds = {
    name: final_points[final_points["модель"] == f"{name}, финальная конфигурация"].set_index("правило")["порог"][evaluate.MAIN_RULES].to_dict()
    for name in SERVICE
}
frozen = finalist.freeze(finalist.service_record({name: final_models[name] for name in SERVICE}, X_train, y_train, service_thresholds))
print("заморожено", frozen["дата"], "отпечаток данных", frozen["данные"])
pd.DataFrame({name: record["пороги"] for name, record in frozen["модели"].items()})

### 11.1 Различение и калибровка на тесте

In [ ]:
test_scores = {name: final_models[name].predict_proba(X_test)[:, 1] for name in SERVICE}
test_table = pd.DataFrame([evaluate.test_metrics(name, scores, y_test) for name, scores in test_scores.items()])
# ДИ OOF финальной конфигурации на train для сравнения
reference = final_summary.set_index("модель")
for name in SERVICE:
    row = reference.loc[f"{name}, финальная конфигурация"]
    test_table.loc[test_table["модель"] == name, "OOF train"] = row["ROC-AUC OOF"]
    test_table.loc[test_table["модель"] == name, "OOF ДИ низ"] = row["ROC-AUC ДИ низ"]
    test_table.loc[test_table["модель"] == name, "OOF ДИ верх"] = row["ROC-AUC ДИ верх"]
test_table["внутри ДИ OOF"] = test_table["ROC-AUC"].between(test_table["OOF ДИ низ"], test_table["OOF ДИ верх"])
print(f"тест: {len(y_test)} пациентов, рецидивов {y_test.sum()}")
save_table(test_table.round(3), "step11_test")
test_table.round(3).T

In [ ]:
test_results = [evaluate.test_result(name, scores, y_test) for name, scores in test_scores.items()]
fig = evaluate.plot_curves(test_results, "тест")
caption(fig, f"ROC- и PR-кривые логрега Б и CatBoost на тесте ({len(y_test)} пациентов, {y_test.sum()} рецидивов). Пунктир слева - случайное угадывание, справа - доля рецидива на тесте.")
save_figure(fig, "step11_curves")
plt.show()

In [ ]:
fig = evaluate.plot_calibration(test_results, "тест")
caption(fig, "Калибровка логрега Б и CatBoost на тесте: средний прогноз и наблюдаемая доля рецидива в 5 квантильных группах прогнозов (около 12 пациентов в группе). Пунктир - идеальная калибровка.")
save_figure(fig, "step11_calibration")
plt.show()

### 11.2 Замороженные пороги на тесте

In [ ]:
test_points = pd.concat(
    [evaluate.test_points(name, test_scores[name], y_test, frozen["модели"][name]["пороги"]) for name in SERVICE],
    ignore_index=True,
)
test_points["выборка"] = "тест"
oof_points = fixed_points[fixed_points["модель"].isin([f"{name}, финальная конфигурация" for name in SERVICE])].copy()
oof_points["модель"] = oof_points["модель"].str.removesuffix(", финальная конфигурация")
oof_points["выборка"] = "OOF train"
threshold_compare = pd.concat([test_points, oof_points], ignore_index=True).sort_values(["модель", "порог", "выборка"], ascending=[True, True, False])
save_table(threshold_compare.round(3), "step11_thresholds")
threshold_compare.set_index(["модель", "порог", "выборка"]).round(3).T

In [ ]:
fig = evaluate.plot_dca(test_results, "тест")
caption(fig, "Кривые принятия решений логрега Б и CatBoost на тесте против стратегий «пометить всех» и «никого».")
save_figure(fig, "step11_dca")
plt.show()
pd.DataFrame([evaluate.useful_thresholds(r) for r in test_results])

### Вывод шага 11

- Тест открыт один раз, после заморозки моделей и порогов (запись от 2026-09-27, отпечаток данных f8346e788e494d18-f1add192113b26ff). 58 пациентов, 18 рецидивов.
- Логрег Б: ROC-AUC 0,799 (95% ДИ 0,669-0,915), PR-AUC 0,686 (0,477-0,861). ROC-AUC внутри ДИ OOF финальной конфигурации на train (0,688-0,816).
- CatBoost: ROC-AUC 0,662 (0,516-0,805), PR-AUC 0,489 (0,291-0,707). Тоже внутри ДИ OOF на train (0,656-0,781).
- ДИ двух моделей на тесте перекрываются: при 18 рецидивах тест не различает их. Разница точечных оценок (0,799 против 0,662) - в пределах шума теста.
- Калибровка (рисунок 40): логрег Б - наклон 2,403, прогнозы сильно сжаты к средней доле, масштабированный Brier 0,200. CatBoost - наклон 0,993, масштабированный Brier 0,072. Калибровка в среднем 0,059 и 0,025, средний прогноз близок к доле рецидива. Наклон на 58 пациентах оценен грубо.
- Замороженный порог F1 (помечено, TP, FP, TN, FN, чувствительность, специфичность, PPV, NPV), рядом OOF на train при том же пороге:
  - логрег Б, порог 0,287: 52%; 14, 16, 24, 4; 0,778; 0,600; 0,467; 0,857. OOF: чувствительность 0,806, специфичность 0,544, PPV 0,443, NPV 0,862;
  - CatBoost, порог 0,304: 53%; 13, 18, 22, 5; 0,722; 0,550; 0,419; 0,815. OOF: 0,856; 0,519; 0,445; 0,890.
- Порог F2:
  - логрег Б, 0,219: 74%; 17, 26, 14, 1; 0,944; 0,350; 0,395; 0,933;
  - CatBoost, 0,206: 76%; 16, 28, 12, 2; 0,889; 0,300; 0,364; 0,857.
- Метрики при порогах близки к OOF на train, у CatBoost чувствительность на тесте ниже (0,722 против 0,856, то есть 5 пропущенных рецидивов из 18 вместо ожидаемых примерно 2,6).
- Кривая принятия решений (рисунок 41) выше обеих стратегий: логрег Б при порогах 0,09-0,60, CatBoost в основном 0,12-0,35 и 0,43-0,53.
- Итог: оценки на тесте согласуются с валидацией на train, обе модели переносятся на новых пациентов. Модели и пороги после теста не меняем.

## Приложение А. TabPFN (в статью не идет)

TabPFN v2 - нейросеть, заранее обученная на миллионах синтетических таблиц. На наших данных она не учится в обычном смысле: обучающая часть фолда подается ей как контекст, и прогноз для новых пациентов получается сразу. Гиперпараметры не подбираем, поэтому ее оценка по 25 фолдам не завышена выбором. Пропуски модель обрабатывает сама, номинальные признаки объявлены категориальными. Остальные признаки TabPFN проверяет сам: столбец с менее чем 4 разными значениями (пропуск считается значением) тоже идет категориальным, так ведут себя все бинарные и часть порядковых. Автоопределение оставлено как в модели по умолчанию. Считает локально на процессоре, данные пациентов никуда не уходят (`models.tabpfn`).

Сравниваем с логрегом Б, CatBoost и случайным лесом.

In [ ]:
tabpfn_result = evaluate.cross_validate("TabPFN v2", models.tabpfn(), X_train, y_train)
tabpfn_results = [logreg_vote, tabpfn_result] + trees
tabpfn_summary = evaluate.summary_table(tabpfn_results)
save_table(tabpfn_summary.round(3), "appendixA_models")
tabpfn_summary.round(3).T

In [ ]:
tabpfn_points = pd.concat([evaluate.operating_points(r) for r in tabpfn_results[:2]], ignore_index=True)
save_table(tabpfn_points.round(3), "appendixA_operating_points")
tabpfn_points[tabpfn_points["правило"].isin(evaluate.MAIN_RULES)].round(3).T

In [ ]:
fig = evaluate.plot_cv_auc(tabpfn_summary)
caption(fig, "ROC-AUC OOF с 95% бутстрэп-ДИ: логрег Б, TabPFN v2 без подбора, CatBoost и случайный лес с подбором Optuna.")
save_figure(fig, "appendixA_cv_auc")
plt.show()

In [ ]:
fig = evaluate.plot_curves(tabpfn_results[:2])
caption(fig, "ROC- и PR-кривые логрега Б и TabPFN v2, OOF-прогнозы на train, все 5 повторов CV вместе.")
save_figure(fig, "appendixA_curves")
plt.show()

In [ ]:
fig = evaluate.plot_calibration(tabpfn_results[:2])
caption(fig, "Калибровка логрега Б и TabPFN v2: средний прогноз и наблюдаемая доля рецидива в 5 квантильных группах OOF-прогнозов всех 5 повторов. Пунктир - идеальная калибровка.")
save_figure(fig, "appendixA_calibration")
plt.show()

In [ ]:
fig = evaluate.plot_dca(tabpfn_results[:2])
caption(fig, "Кривые принятия решений логрега Б и TabPFN v2 против стратегий «пометить всех» и «никого».")
save_figure(fig, "appendixA_dca")
plt.show()
pd.DataFrame([evaluate.useful_thresholds(r) for r in tabpfn_results[:2]])

### Вывод приложения А

- TabPFN v2 без подбора: ROC-AUC CV 0,687 ± 0,076, OOF 0,677 (95% ДИ 0,608-0,740). Логрег Б 0,700 (0,633-0,764), CatBoost 0,681, лес 0,685. ДИ перекрываются, прироста нет (рисунок 42).
- PR-AUC у TabPFN ниже всех: 0,424 против 0,489 у логрега Б. Калибровка: наклон 0,816, масштабированный Brier 0,091.
- Рабочая точка F1: порог 0,244; помечено 70%; чувствительность 0,915; специфичность 0,397; PPV 0,408; NPV 0,922. Кривая принятия решений выше обеих стратегий на 0,05-0,43, у логрега Б 0,05-0,52 (рисунок 45).
- Итог: TabPFN результат не меняет, в статью не идет.

## Приложение Б. Генеративная синтетика (в статью не идет)

Станет ли модель лучше, если дописать в обучающую часть синтетических пациентов (`src/synthetic.py`). Генератор учится только на обучающей части фолда, проверка на реальных пациентах валидационной части. Пропуски перед обучением генератора заполняются как в модели (медиана и мода). Три генератора:
- своя гауссова копула, для каждого класса отдельно: распределение каждого признака и корреляции между ними, у бинарных, порядковых и номинальных только реальные коды;
- гауссова копула из пакета SDV: тот же метод в чужой реализации, для перекрестной проверки;
- CTGAN из SDV: нейросеть-генератор, может уловить нелинейные связи, но на малых данных нестабилен.

Генераторы SDV учатся на всей обучающей части вместе с исходом, пациенты нужного класса порождаются по условию на исход. Настройки SDV по умолчанию: копула описывает каждый количественный признак бета-распределением, CTGAN учится 300 эпох пакетами по 500 строк, при 183 пациентах фолда это один шаг обучения на эпоху. Зерно выборки SDV задаем сами (иначе SDV берет свое постоянное). Лицензия SDV (BUSL-1.1) разрешает такое использование, запрещен только коммерческий сервис синтетических данных.

Модель - логрег Б в финальной конфигурации (набор голосования и параметры сетки на всем train, 9.1). Набор и параметры фиксированы во всех вариантах, поэтому оценки не вложены, но варианты между собой сравнимы. Режимы:
- удвоение: в каждом классе столько же синтетических, сколько реальных, доля рецидивов не меняется;
- баланс: синтетические рецидивы до равенства классов, как SMOTENC на шаге 5.

CTGAN учится около 20 секунд на фолд, поэтому результаты вариантов сохраняются в `data/interim/step11` и при тех же данных и настройках читаются с диска.

In [ ]:
quality_ks, quality_pairs = [], {}
for generator in synthetic.GENERATORS:
    ks, pairs = synthetic.quality(X_train, y_train, generator)
    quality_ks.append(ks)
    quality_pairs[generator] = pairs
quality_ks = pd.concat(quality_ks, ignore_index=True)
save_table(quality_ks.round(3), "appendixB_ks")
fig = synthetic.plot_correlation_pairs(quality_pairs)
caption(fig, "Корреляции Спирмена всех пар признаков у реальных пациентов train (заполненных медианой и модой) и у синтетических трех генераторов, обученных на них же. Пунктир - полное совпадение, в заголовке - корреляция между двумя наборами корреляций. Сравнение с теми же пациентами, на которых генератор учился, поэтому сходство оптимистично.")
save_figure(fig, "appendixB_correlation_pairs")
plt.show()
quality_ks.groupby("генератор")["KS"].agg(["median", "max"]).round(3)

In [ ]:
final_logreg = final_builds["логрег Б"]
synth_results = [evaluate.cross_validate("логрег, без синтетики", final_logreg, X_train, y_train)]
for generator in synthetic.GENERATORS:
    for mode in ["удвоение", "баланс"]:
        name = f"{generator}, {mode}"
        augmented = synthetic.SyntheticAugment(final_logreg, mode, generator)
        synth_results.append(synthetic.cross_validate_cached(name, augmented, X_train, y_train))
synth_summary = evaluate.summary_table(synth_results)
save_table(synth_summary.round(3), "appendixB_models")
synth_summary.round(3).T

In [ ]:
synth_points = pd.concat([evaluate.operating_points(r) for r in synth_results], ignore_index=True)
save_table(synth_points.round(3), "appendixB_operating_points")
synth_points[synth_points["правило"] == "максимум F1"].round(3).T

In [ ]:
fig = evaluate.plot_cv_auc(synth_summary)
caption(fig, "ROC-AUC OOF с 95% бутстрэп-ДИ финального логрега (параметры фиксированы) без синтетики и с синтетикой трех генераторов в двух режимах: удвоение обучающей части и баланс классов.")
save_figure(fig, "appendixB_cv_auc")
plt.show()

In [ ]:
calibration_shown = [synth_results[0]] + [r for r in synth_results if r["модель"].endswith("удвоение")]
fig = evaluate.plot_calibration(calibration_shown)
caption(fig, "Калибровка финального логрега без синтетики и с удвоением обучающей части синтетикой трех генераторов: средний прогноз и наблюдаемая доля рецидива в 5 квантильных группах OOF-прогнозов всех 5 повторов. Пунктир - идеальная калибровка.")
save_figure(fig, "appendixB_calibration")
plt.show()

### Вывод приложения Б

- Сходство синтетики с реальными пациентами, медиана KS по количественным признакам: своя копула 0,048, копула SDV 0,218, CTGAN 0,306. Своя копула воспроизводит распределения лучше всех. Корреляции пар признаков - на рисунке 46.
- Логрег Б в финальной конфигурации, ROC-AUC OOF (рисунок 47): без синтетики 0,755 (95% ДИ 0,688-0,816). Своя копула 0,753 при удвоении и 0,761 (0,695-0,823) при балансе, копула SDV 0,739 и 0,736, CTGAN 0,750 и 0,722. ДИ всех вариантов перекрываются с вариантом без синтетики.
- Баланс сдвигает прогнозы вверх, как балансировка на шаге 5: калибровка в среднем от -0,762 до -0,796, масштабированный Brier 0,052, -0,010 и -0,016 против 0,147 без синтетики.
- Итог: синтетика модель не улучшает, в статью не идет. Оценки не вложены (набор и параметры фиксированы), варианты сравнимы только между собой.